# Weight Initialization Analysis

Analyze weight properties: scale profiles, distribution statistics,
norm comparisons, sparsity, and embedding weight analysis.

## Why This Matters

Weight initialization examines the structure and statistics of model parameters. The structure of weights constrains what computations a component can perform, and spectral analysis can reveal functional specialization, redundancy, and low-rank structure.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.weight_initialization_analysis import (
    weight_scale_profile, weight_distribution_stats,
    weight_norm_comparison, weight_sparsity_profile,
    embedding_weight_analysis,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
print('Model ready')

## Weight Scale Profile

In [ ]:
result = weight_scale_profile(model)
print(f"{result['n_weight_matrices']} weight matrices, mean std: {result['mean_std']:.4f}\n")
for w in result['per_weight'][:12]:
    print(f"  {w['name']:12s} {str(w['shape']):20s}: "
          f"norm={w['mean_norm']:.4f}, std={w['std']:.4f}, max={w['max_abs']:.4f}")

## Distribution Statistics

In [ ]:
result = weight_distribution_stats(model, layer=0)
print(f"Layer {result['layer']}: {result['total_params']} params\n")
for w in result['per_weight']:
    print(f"  {w['name']:5s}: mean={w['mean']:+.4f}, std={w['std']:.4f}, "
          f"skew={w['skewness']:+.3f}, kurt={w['kurtosis']:.3f}, "
          f"near_zero={w['near_zero_fraction']:.1%}")

## Norm Comparison

In [ ]:
result = weight_norm_comparison(model)
bal = 'BALANCED' if result['is_balanced'] else 'IMBALANCED'
print(f"Max ratio: {result['max_layer_ratio']:.4f} [{bal}]\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: total={p['total_norm']:.4f}, "
          f"attn={p['attn_norm']:.4f}, mlp={p['mlp_norm']:.4f} "
          f"(attn_frac={p['attn_fraction']:.2f})")

## Sparsity Profile

In [ ]:
result = weight_sparsity_profile(model)
print(f"Total sparsity: {result['total_sparsity']:.2%} (threshold={result['threshold']})\n")
for p in result['per_layer']:
    bar = '█' * int(p['sparsity'] * 40)
    print(f"  Layer {p['layer']}: {p['n_sparse']}/{p['n_params']} "
          f"({p['sparsity']:.1%}) {bar}")

## Embedding Weight Analysis

In [ ]:
result = embedding_weight_analysis(model)
iso = 'ISOTROPIC' if result['is_isotropic'] else 'anisotropic'
print(f"Embed norm: {result['embed_mean_norm']:.4f} (CV={result['embed_norm_cv']:.4f})")
print(f"Unembed norm: {result['unembed_mean_norm']:.4f} (CV={result['unembed_norm_cv']:.4f})")
print(f"E-U alignment: {result['embed_unembed_alignment']:.4f}")
print(f"Isotropy: [{iso}]")